In [ ]:
# Figure 7C-G: synapses and matched neuronal cable
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from scipy.stats import ks_2samp, norm, wilcoxon
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

matched_file = Path('data') / 'matched_synapse_cable.csv'
out_dir = Path('fig7_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
nt_labels = {'ACH': 'ACh', 'GABA': 'GABA', 'GLUT': 'GLUT', 'DA': 'DA', 'SER': 'SER', 'OCT': 'OCT'}
nt_colors = {'ACH': '#E82C2C', 'GABA': '#E79A24', 'GLUT': '#237A20',
             'DA': '#3037E8', 'SER': '#7B1C83', 'OCT': '#A4483E'}
super_order = ['OL', 'MB', 'CX', 'LX', 'VLNP', 'LH', 'SNP', 'INP',
               'AL', 'VMNP', 'PENP', 'GNG', 'OCG']
min_matches = 100
n_bootstrap = 5000
seed = 20260818

plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'axes.labelsize': 10,
                     'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8,
                     'axes.linewidth': 0.8, 'svg.fonttype': 'none'})

# Matched presynaptic sites and their three non-presynaptic controls
cols = ['match_id', 'pre_root_id', 'is_synapse', 'distance_to_trachea_surface_um',
        'nt_type', 'super_region', 'neuropil']
matched = pd.read_csv(matched_file, usecols=cols,
                      dtype={'pre_root_id': 'string', 'nt_type': 'string',
                             'super_region': 'string', 'neuropil': 'string'})
matched = matched.loc[~matched['neuropil'].isin(['LA_L', 'LA_R'])]
matched['nt_type'] = matched['nt_type'].str.strip().str.upper().replace({
    'ACETYLCHOLINE': 'ACH', 'GLUTAMATE': 'GLUT', 'DOPAMINE': 'DA',
    'SEROTONIN': 'SER', '5HT': 'SER', '5-HT': 'SER', 'OCTOPAMINE': 'OCT'}).fillna('UNKNOWN')
matched['super_region'] = matched['super_region'].str.strip().fillna('UNKNOWN')
matched = matched.sort_values(['match_id', 'is_synapse'], ascending=[True, False], kind='stable')

n_matches = len(matched) // 4
distance4 = matched['distance_to_trachea_surface_um'].to_numpy(dtype=float).reshape(n_matches, 4)
meta = matched.iloc[::4][['pre_root_id', 'nt_type', 'super_region']].reset_index(drop=True)
del matched
synapse_distance = distance4[:, 0]
cable_distance = distance4[:, 1:].reshape(-1)


def ecdf(values, grid):
    sorted_values = np.sort(np.asarray(values, dtype=float))
    return np.searchsorted(sorted_values, grid, side='right') / len(sorted_values)


def conditional_logit(distances):
    # Exact conditional likelihood for one case and three controls per stratum.
    x = np.asarray(distances, dtype=float) / 5.0

    def negative_log_likelihood(beta):
        scores = beta * x
        return float(np.sum(logsumexp(scores, axis=1) - scores[:, 0]))

    beta = float(minimize_scalar(negative_log_likelihood, bounds=(-6, 6),
                                 method='bounded', options={'xatol': 1e-10}).x)
    scores = beta * x
    weights = np.exp(scores - logsumexp(scores, axis=1)[:, None])
    expected = np.sum(weights * x, axis=1)
    information = np.sum(np.sum(weights * x**2, axis=1) - expected**2)
    se = 1.0 / np.sqrt(information)
    p_value = 2.0 * norm.sf(abs(beta / se))
    return np.exp(-beta), p_value


# Figure 7C: overall synapse and matched-cable ECDF
x = np.linspace(0, 40, 1601)
synapse_cdf = ecdf(synapse_distance, x)
cable_cdf = ecdf(cable_distance, x)
ks = ks_2samp(synapse_distance, cable_distance, method='asymp')
signed_d = float(ks.statistic * ks.statistic_sign)
overall_or, overall_p = conditional_logit(distance4)

site_delta = synapse_distance - np.median(distance4[:, 1:], axis=1)
neuron_delta = pd.DataFrame({'neuron': meta['pre_root_id'], 'delta_um': site_delta})
neuron_delta = neuron_delta.groupby('neuron')['delta_um'].median().to_numpy()
median_neuron_delta = float(np.median(neuron_delta))
rng = np.random.default_rng(seed)
bootstrap = np.array([np.median(rng.choice(neuron_delta, len(neuron_delta), replace=True))
                      for _ in range(n_bootstrap)])
ci_low, ci_high = np.percentile(bootstrap, [2.5, 97.5])
neuron_p = wilcoxon(neuron_delta).pvalue

fig, ax = plt.subplots(figsize=(4.5, 3.3))
ax.plot(x, synapse_cdf, color='#2878B5', linewidth=2, label='Presynaptic sites')
ax.plot(x, cable_cdf, color='#E28B37', linewidth=2, label='Matched cable')
ax.set(xlim=(0, 40), ylim=(0, 1), xlabel='Distance to tracheole surface (µm)',
       ylabel='Cumulative fraction')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(out_dir / 'fig7c.svg', bbox_inches='tight')
plt.show()

print(f"Figure 7C: {n_matches:,} matched sites, {meta['pre_root_id'].nunique():,} neurons")
print(f'Figure 7C: synapse median = {np.median(synapse_distance):.3f} µm; '
      f'cable median = {np.median(cable_distance):.3f} µm')
print(f'Figure 7C: OR per 5 µm closer = {overall_or:.6f}, P = {overall_p:.3g}')
print(f'Figure 7C: neuron-level median displacement = {median_neuron_delta:+.3f} µm '
      f'(95% CI {ci_low:+.3f} to {ci_high:+.3f}), Wilcoxon P = {neuron_p:.3g}')


# Figure 7D: signed difference between the matched ECDFs
fig, ax = plt.subplots(figsize=(4.5, 3.3))
ax.axhline(0, color='0.6', linewidth=0.9, linestyle='--')
ax.plot(x, synapse_cdf - cable_cdf, color='#2A2A2A', linewidth=2)
ax.set(xlim=(0, 40), xlabel='Distance to tracheole surface (µm)',
       ylabel='ΔECDF (synapse − matched cable)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(out_dir / 'fig7d.svg', bbox_inches='tight')
plt.show()
print(f'Figure 7D: signed KS D = {signed_d:+.6f} at {ks.statistic_location:.3f} µm')


# Figure 7E: site-level displacement relative to local neuronal cable
from scipy.stats import spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

local_cable_distance = np.median(distance4[:, 1:], axis=1)
synaptic_displacement = site_delta
site_rho, site_p = spearmanr(local_cable_distance, synaptic_displacement)

neuron_summary = pd.DataFrame({'neuron': meta['pre_root_id'],
                               'local_cable_distance': local_cable_distance,
                               'synaptic_displacement': synaptic_displacement})
neuron_summary = neuron_summary.groupby('neuron')[['local_cable_distance', 'synaptic_displacement']].median()
neuron_rho, neuron_p = spearmanr(neuron_summary['local_cable_distance'],
                                neuron_summary['synaptic_displacement'])

rng_e = np.random.default_rng(seed)
fit_indices = rng_e.choice(n_matches, size=min(20000, n_matches), replace=False)
plot_indices = rng_e.choice(n_matches, size=min(1000, n_matches), replace=False)
x_fit = local_cable_distance[fit_indices]
y_fit = synaptic_displacement[fit_indices]
cluster = pd.factorize(meta['pre_root_id'].to_numpy()[fit_indices])[0]
n_clusters = cluster.max() + 1

x_grid = np.linspace(0, 25, 200)
frac = 0.35
n_bootstrap_e = 500
fit = lowess(y_fit, x_fit, frac=frac, it=1, delta=0.15, return_sorted=True)
trend = np.interp(x_grid, fit[:, 0], fit[:, 1])

bin_edges = np.quantile(local_cable_distance[local_cable_distance <= 20], np.linspace(0, 1, 11))
bins = np.searchsorted(bin_edges[1:-1], local_cable_distance, side='right')
bins_fit = np.searchsorted(bin_edges[1:-1], x_fit, side='right')
bin_x = np.array([np.median(local_cable_distance[bins == i]) for i in range(10)])
bin_y = np.array([np.median(synaptic_displacement[bins == i]) for i in range(10)])

boot_trend = np.empty((n_bootstrap_e, len(x_grid)))
boot_bins = np.empty((n_bootstrap_e, len(bin_y)))
for j in range(n_bootstrap_e):
    weights = rng_e.multinomial(n_clusters, np.full(n_clusters, 1 / n_clusters))
    indices = np.repeat(np.arange(len(x_fit)), weights[cluster])
    xb, yb = x_fit[indices], y_fit[indices]
    smooth = lowess(yb, xb, frac=frac, it=1, delta=0.15, return_sorted=True)
    boot_trend[j] = np.interp(x_grid, smooth[:, 0], smooth[:, 1])
    bb = bins_fit[indices]
    boot_bins[j] = [np.median(yb[bb == i]) for i in range(10)]

trend_lo, trend_hi = np.percentile(boot_trend, [2.5, 97.5], axis=0)
bin_lo, bin_hi = np.percentile(boot_bins, [2.5, 97.5], axis=0)

fig, ax = plt.subplots(figsize=(4.7, 3.5))
ax.scatter(local_cable_distance[plot_indices], synaptic_displacement[plot_indices],
           s=7, color='#AEB6BE', alpha=0.15, edgecolors='none', rasterized=True)
ax.axhline(0, color='0.6', linewidth=0.8, linestyle='--')
ax.fill_between(x_grid, trend_lo, trend_hi, color='#B7C4CD', alpha=0.38, linewidth=0)
ax.plot(x_grid, trend, color='#263E50', linewidth=2)
ax.errorbar(bin_x, bin_y, yerr=[bin_y - bin_lo, bin_hi - bin_y],
            fmt='o-', color='#E13C30', markersize=4.5, linewidth=1,
            elinewidth=0.9, capsize=2, markeredgecolor='white', markeredgewidth=0.5)
ax.set(xlim=(0, 25), xlabel='Local neuronal cable distance to tracheole (µm)',
       ylabel='Synapse displacement relative to local cable (µm)')
ax.spines[['top', 'right']].set_visible(False)
p_label = 'P < 0.001' if site_p < 0.001 else f'P = {site_p:.3g}'
ax.text(0.97, 0.97, f'ρ = {site_rho:.3f}\n{p_label}', transform=ax.transAxes,
        ha='right', va='top')
fig.tight_layout()
fig.savefig(out_dir / 'fig7e.svg', bbox_inches='tight')
plt.show()
print(f'Figure 7E: {n_matches:,} sites, Spearman ρ = {site_rho:.6f}, P = {site_p:.6g}')
print(f'Figure 7E: {len(neuron_summary):,} neurons, Spearman ρ = {neuron_rho:.6f}, P = {neuron_p:.6g}')


# Figure 7F: neurotransmitter-specific ΔECDF
x_nt = np.linspace(0, 25, 1501)
nt_curves = {}
nt_stats = []
for nt in nt_order:
    mask = meta['nt_type'].eq(nt).to_numpy()
    syn = distance4[mask, 0]
    cable = distance4[mask, 1:].reshape(-1)
    difference = ecdf(syn, x_nt) - ecdf(cable, x_nt)
    index = np.argmax(np.abs(difference))
    nt_curves[nt] = difference
    nt_stats.append((nt, int(mask.sum()), difference[index], x_nt[index]))

ymax = max(0.02, np.ceil(max(np.abs(v).max() for v in nt_curves.values()) / 0.01) * 0.01)
fig, ax = plt.subplots(figsize=(5.2, 3.7))
ax.axhline(0, color='0.5', linewidth=0.9, linestyle='--')
for nt, n, effect, location in nt_stats:
    ax.plot(x_nt, nt_curves[nt], color=nt_colors[nt], linewidth=2,
            label=f'{nt_labels[nt]}  D = {effect:+.3f}')
ax.set(xlim=(0, 25), ylim=(-ymax, ymax), xlabel='Distance to tracheole surface (µm)',
       ylabel='ΔECDF (synapse − matched cable)')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, loc='best')
fig.tight_layout()
fig.savefig(out_dir / 'fig7f.svg', bbox_inches='tight')
plt.show()
print('Figure 7F: NT, matched sites, signed D, distance (µm)')
for nt, n, effect, location in nt_stats:
    print(f'  {nt_labels[nt]:>4}: {n:>7,}, D = {effect:+.4f}, {location:.2f} µm')


# Figure 7G: super-region × neurotransmitter conditional logistic regression
rows = []
for region in super_order:
    for nt in nt_order:
        mask = (meta['super_region'].eq(region) & meta['nt_type'].eq(nt)).to_numpy()
        n = int(mask.sum())
        if n < min_matches:
            continue
        odds_ratio, p_value = conditional_logit(distance4[mask])
        rows.append({'super_region': region, 'nt_type': nt, 'n_matches': n,
                     'or_per_5um_closer': odds_ratio, 'log2_or': np.log2(odds_ratio),
                     'p_value': p_value})

results = pd.DataFrame(rows)
results['q_fdr'] = multipletests(results['p_value'], method='fdr_bh')[1]
results['significant'] = results['q_fdr'] < 0.05

color_limit = max(0.25, results['log2_or'].abs().max())
cmap = plt.get_cmap('RdBu_r')
color_norm = TwoSlopeNorm(vmin=-color_limit, vcenter=0, vmax=color_limit)
root_n = np.sqrt(results['n_matches'])
results['marker_size'] = 28 + 322 * (root_n - root_n.min()) / (root_n.max() - root_n.min())

fig = plt.figure(figsize=(7.2, 6.35))
ax = fig.add_axes([0.16, 0.09, 0.59, 0.84])
for row in results.itertuples(index=False):
    ax.scatter(nt_order.index(row.nt_type), super_order.index(row.super_region),
               s=row.marker_size, color=cmap(color_norm(row.log2_or)), alpha=0.96,
               edgecolor='#303030' if row.significant else 'none',
               linewidth=0.95 if row.significant else 0)
ax.set(xlim=(-0.55, len(nt_order) - 0.45), ylim=(-0.6, len(super_order) - 0.4),
       xticks=np.arange(len(nt_order)), yticks=np.arange(len(super_order)),
       xticklabels=[nt_labels[nt] for nt in nt_order], yticklabels=super_order,
       ylabel='Super-region')
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(handles=[Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='white',
                          markeredgecolor='#303030', markersize=8, label='FDR < 0.05')],
          loc='upper left', bbox_to_anchor=(1.02, 1.01), frameon=False)
color_axis = fig.add_axes([0.79, 0.20, 0.028, 0.60])
colorbar = fig.colorbar(ScalarMappable(norm=color_norm, cmap=cmap), cax=color_axis)
colorbar.set_label('log2 OR', rotation=270, labelpad=16)
fig.text(0.845, 0.755, 'Synapse closer than cable', fontsize=8.5)
fig.text(0.845, 0.245, 'Cable closer than synapse', fontsize=8.5)
fig.savefig(out_dir / 'fig7g.svg', bbox_inches='tight')
plt.show()
print(f"Figure 7G: {int(results['significant'].sum())} of {len(results)} region × NT "
      'combinations significant after BH-FDR')